# 🎵 Music Streaming Database Analytics (SQLite)

## 📚 Course Information
- **Course:** CSCE 5350 – Fundamentals of Database Systems  
- **Assignment:** Project Group 7  
- **Contribution Student Name:** Noah Khan

---

## 📌 Project Overview
This notebook contains the implementation and analysis of a relational database for a music streaming platform using **SQLite**.

The objective of this work is to:
- Design and initialize a normalized database schema
- Populate the database with sample data
- Execute analytical SQL queries to extract meaningful insights

---

## 🧠 Notes
- All queries are written in standard SQL compatible with SQLite.
- The dataset is synthetic and intended for academic purposes.
- Each section of the notebook is organized to reflect a real-world analytics workflow.

---

## 🚀 How to Use This Notebook
1. Run the setup cells to initialize the database  
2. Execute each query cell to view results  
3. Review outputs and insights for analysis  

In [1]:
import sqlite3
import pandas as pd

In [2]:
# load the database with the complete toy data
conn = sqlite3.connect("Music_Streaming_Service.db")
cursor = conn.cursor()

In [3]:
def run_query(query):
    return pd.read_sql_query(query, conn)

### The Top 10 Most Recent Albums from Different Artist

In [4]:
run_query("""
    WITH RankedAlbums AS (
    SELECT 
        album_id,
        artist_id,
        title AS album_title,
        release_date,
        ROW_NUMBER() OVER(PARTITION BY artist_id ORDER BY release_date DESC) as rn
    FROM Album
)
SELECT 
    a.album_title,
    art.name AS artist_name,
    a.release_date
FROM RankedAlbums a
JOIN Artist art ON a.artist_id = art.artist_id
WHERE a.rn = 1
ORDER BY a.release_date DESC
LIMIT 10;
""")

,album_title,artist_name,release_date
0,beerbongs & bentleys,Post Malone,2018
1,25,Adele,2015
2,Cheese,Stromae,2010
3,Hands All Over (Deluxe Edition),Maroon 5,2010
4,Loud (Deluxe),Rihanna,2010
5,When the World Comes Down,The All-American Rejects,2008
6,Circus,Britney Spears,2008
7,Franz Ferdinand (Special Edition Version),Franz Ferdinand,2004
8,Fallen,Evanescence,2003
9,Meteora (Deluxe Version),LINKIN PARK,2003


### Most Popular Artist for Each User

In [5]:
run_query("""
    WITH UserArtistCounts AS (
    SELECT 
        u.user_id,
        u.username,
        art.name AS artist_name,
        COUNT(ps.song_id) AS appearance_count,
        RANK() OVER(PARTITION BY u.user_id ORDER BY COUNT(ps.song_id) DESC) as rnk
    FROM User u
    JOIN Playlist p ON u.user_id = p.user_id
    JOIN PlaylistSong ps ON p.playlist_id = ps.playlist_id
    JOIN Song s ON ps.song_id = s.song_id
    JOIN Album a ON s.album_id = a.album_id
    JOIN Artist art ON a.artist_id = art.artist_id
    GROUP BY 
        u.user_id, 
        u.username, 
        art.artist_id, 
        art.name
)
SELECT 
    username,
    artist_name,
    appearance_count
FROM UserArtistCounts
WHERE rnk = 1;        
""")

,username,artist_name,appearance_count
0,mgarcia_dev,Adele,4
1,mgarcia_dev,Britney Spears,4
2,mgarcia_dev,Post Malone,4
3,mgarcia_dev,Stevie Nicks,4
4,bobbyc,Michael Jackson,2
5,bobbyc,Adele,2
6,bobbyc,Post Malone,2
7,elena_r,Adele,4
8,elena_r,The All-American Rejects,4
9,elena_r,Berlin,4


### Most Frequently Added Artist in Playlists per Region

In [6]:
run_query("""
    WITH RegionArtistCounts AS (
    SELECT 
        r.region AS region_name,
        art.name AS artist_name,
        COUNT(ps.song_id) AS add_count,
        RANK() OVER(PARTITION BY r.region_id ORDER BY COUNT(ps.song_id) DESC) as rnk
    FROM Region r
    JOIN UserRegion ur ON r.region_id = ur.region_id
    JOIN Playlist p ON ur.user_id = p.user_id
    JOIN PlaylistSong ps ON p.playlist_id = ps.playlist_id
    JOIN Song s ON ps.song_id = s.song_id
    JOIN Album a ON s.album_id = a.album_id
    JOIN Artist art ON a.artist_id = art.artist_id
    GROUP BY 
        r.region_id, 
        r.region, 
        art.artist_id, 
        art.name
)
SELECT 
    region_name,
    artist_name,
    add_count
FROM RegionArtistCounts
WHERE rnk = 1;         
""")

,region_name,artist_name,add_count
0,United States,Michael Jackson,16
1,United Kingdom,Maroon 5,16
2,Germany,Adele,4
3,Germany,The All-American Rejects,4
4,Germany,Berlin,4
5,Germany,Evanescence,4
6,Japan,Michael Jackson,5
7,Brazil,Michael Jackson,6
8,France,LINKIN PARK,3
9,Mexico,Adele,10


### Count of Subscripton Type per Region

In [7]:
run_query("""
    SELECT 
    r.region AS region_name,
    sp.type AS subscription_type,
    COUNT(us.user_subscription_id) AS total_subscriptions
FROM Region r
JOIN UserRegion ur ON r.region_id = ur.region_id
JOIN UserSubscription us ON ur.user_id = us.user_id
JOIN SubscriptionPlan sp ON us.subscription_id = sp.subscription_id
GROUP BY 
    r.region_id,
    r.region,
    sp.subscription_id,
    sp.type
ORDER BY 
    r.region, 
    total_subscriptions DESC;          
""")

,region_name,subscription_type,total_subscriptions
0,Australia,Individual Annual,1
1,Australia,Family Annual,1
2,Brazil,Individual Monthly,3
3,Brazil,Individual Annual,1
4,Brazil,Family Monthly,1
5,Canada,Individual Monthly,1
6,Canada,Individual Annual,1
7,France,Individual Monthly,2
8,Germany,Individual Annual,1
9,Germany,Family Annual,1


### Users that has 3 or Less Playlists Created

In [8]:
run_query("""
    SELECT 
    u.user_id,
    u.username,
    COUNT(p.playlist_id) AS playlist_count
FROM User u
LEFT JOIN Playlist p ON u.user_id = p.user_id
GROUP BY 
    u.user_id,
    u.username
HAVING 
    COUNT(p.playlist_id) <= 3
ORDER BY 
    playlist_count DESC;
""")

,user_id,username,playlist_count
0,2,mgarcia_dev,3
1,6,elena_r,3
2,15,lpetro,3
3,18,adavies,3
4,48,gflores,3
5,3,bobbyc,1
6,7,loconnor,1
7,8,chloe_d,1
8,9,yuki_t,1
9,14,emma_j,1


### Popular Genre per Region

In [9]:
run_query("""
    WITH RegionGenreCounts AS (
    SELECT 
        r.region AS region_name,
        g.name AS genre_name,
        COUNT(ps.song_id) AS appearance_count,
        RANK() OVER(PARTITION BY r.region_id ORDER BY COUNT(ps.song_id) DESC) as rnk
    FROM Region r
    JOIN UserRegion ur ON r.region_id = ur.region_id
    JOIN Playlist p ON ur.user_id = p.user_id
    JOIN PlaylistSong ps ON p.playlist_id = ps.playlist_id
    JOIN SongGenre sg ON ps.song_id = sg.song_id
    JOIN Genre g ON sg.genre_id = g.genre_id
    GROUP BY 
        r.region_id, 
        r.region, 
        g.genre_id, 
        g.name
)
SELECT 
    region_name,
    genre_name,
    appearance_count
FROM RegionGenreCounts
WHERE rnk = 1;        
""")

,region_name,genre_name,appearance_count
0,United States,Pop,39
1,United Kingdom,Pop,51
2,Germany,Pop,16
3,Japan,Pop,13
4,Brazil,Pop,18
5,France,Pop,7
6,Mexico,Pop,26


### Artist that Performs More than One Genre

In [10]:
run_query("""
    SELECT 
    art.artist_id,
    art.name AS artist_name,
    COUNT(DISTINCT sg.genre_id) AS genre_count
FROM Artist art
JOIN Album a ON art.artist_id = a.artist_id
JOIN Song s ON a.album_id = s.album_id
JOIN SongGenre sg ON s.song_id = sg.song_id
GROUP BY 
    art.artist_id,
    art.name
HAVING 
    COUNT(DISTINCT sg.genre_id) > 1
ORDER BY 
    genre_count DESC;          
""")

,artist_id,artist_name,genre_count
0,2,Michael Jackson,3
1,24,Rihanna,3
2,1,The Cranberries,2
3,3,The Proclaimers,2
4,4,Ace of Base,2
5,6,Stromae,2
6,7,a-ha,2
7,8,Adele,2
8,9,The All-American Rejects,2
9,10,Berlin,2


### Monthly Revenue per Region

In [11]:
run_query("""
    SELECT 
    r.region AS region_name,
    ROUND(SUM(sp.price), 2) AS total_monthly_revenue
FROM Region r
JOIN UserRegion ur ON r.region_id = ur.region_id
JOIN UserSubscription us ON ur.user_id = us.user_id
JOIN SubscriptionPlan sp ON us.subscription_id = sp.subscription_id
GROUP BY 
    r.region_id,
    r.region
ORDER BY 
    total_monthly_revenue DESC;         
""")

,region_name,total_monthly_revenue
0,United States,1503.73
1,United Kingdom,432.86
2,Germany,269.98
3,Australia,269.98
4,Japan,243.94
5,Mexico,180.98
6,Brazil,149.95
7,India,121.97
8,Canada,110.98
9,France,21.98


### Average Number of Devices per Subscriber Type

In [12]:
run_query("""
    WITH UserDeviceCounts AS (
    SELECT 
        u.user_id,
        us.subscription_id,
        COUNT(ud.device_id) AS device_count
    FROM User u
    JOIN UserSubscription us ON u.user_id = us.user_id
    LEFT JOIN UserDevice ud ON u.user_id = ud.user_id
    GROUP BY 
        u.user_id, 
        us.subscription_id
)
SELECT 
    sp.type AS subscription_type,
    ROUND(AVG(udc.device_count), 0) AS avg_devices_per_user
FROM UserDeviceCounts udc
JOIN SubscriptionPlan sp ON udc.subscription_id = sp.subscription_id
GROUP BY 
    sp.type,
    sp.subscription_id
ORDER BY 
    avg_devices_per_user DESC;          
""")

,subscription_type,avg_devices_per_user
0,Family Annual,2.0
1,Individual Monthly,2.0
2,Family Monthly,1.0
3,Individual Annual,1.0


### Most Used Device Type

In [13]:
run_query("""
    SELECT 
    d.type AS device_type,
    COUNT(ud.user_device_id) AS usage_count
FROM Device d
JOIN UserDevice ud ON d.device_id = ud.device_id
GROUP BY 
    d.type
ORDER BY 
    usage_count DESC
LIMIT 1;          
""")

,device_type,usage_count
0,Desktop,22
